# CISCaRL **C300** — complete walkthrough (Stage 1 → running)

C300 = the C200 model with the support floor raised to **300 discovery patients**.
Everything below is the real project code (`ciscarl2/`) executed in order.

**Run from the repo root** (the folder that contains `ciscarl2/`). If the packages
are missing: `python -m pip install -r requirements.txt`.


## Stage 0 — Setup / imports

In [ ]:
import os, sys, io, contextlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# locate the repo root (folder containing ciscarl2/)
ROOT = None
for cand in [os.getcwd(), os.path.abspath(".."), "/content/ML-project"]:
    if os.path.isdir(os.path.join(cand, "ciscarl2")):
        ROOT = cand
        break
assert ROOT is not None, "Run from the repo root (folder containing ciscarl2/)"
sys.path.insert(0, ROOT)

from ciscarl2 import dgp, gate23, gate24, gate28
from ciscarl2.gate23 import _config23, _regions, _sim_two, _calibrate_two, TSTAR, _delta_for, E1, E2
from ciscarl2.gate24 import _sim_joint, _calibrate_joint
from ciscarl2.gate26 import _calibrate_kind, _sim_kind
from ciscarl2.gate28 import _regions_kind, _metrics, scre_select_alpha, SurvivalCausalRuleEnsemble
from ciscarl2.estimators import crossfit_nuisances, aipw, phi_on_holdout, ipcw_outcome, truncate_weights
from ciscarl2.gate2 import pseudo_outcome
from ciscarl2.gate20 import _capped_G
from ciscarl2.gate11 import _cart
from ciscarl2.gate22 import _stats
from ciscarl2.gate19 import _leaf_table, _assign_effs, _phi_marginal
from ciscarl2.rules import eval_rule

SEED = 600100
FLOOR = 300          # <-- C300
T_STAR = TSTAR
print("repo root:", ROOT, "| horizon:", T_STAR, "| support floor:", FLOOR)


## Stage 1 — Data + preprocessing
`gate23._config23` loads ACTG175 (arms 0/1), standardizes covariates, sets the
median cut-points, and calibrates the baseline hazard to 73.5% survival.
`gate28._regions_kind` defines the true beneficial region; `gate26._sim_kind`
simulates the semi-synthetic Cox outcome with a known treatment effect.


In [ ]:
cfg = _config23()                       # (pool, mu, sd, cuts, idx, lam0)
lam_c0 = _calibrate_kind(cfg, "single") # set ~10% censoring
pool, mu_, sd_, cuts, idx, lam0 = cfg

rng = np.random.RandomState(SEED)
n = 8000
X = pool[rng.choice(len(pool), n, replace=True)]
R1, R2 = _regions_kind(X, cfg, "single")   # single-region scenario
union = R1 | R2

# simulate a known treatment effect
lp0 = X @ dgp.BETA
s0 = np.exp(-lam0 * T_STAR * np.exp(lp0))
e1 = np.where(R1, _delta_for(E1, s0), 0.0)
e2 = np.where(R2, _delta_for(E2, s0), 0.0)
A = rng.binomial(1, 0.5, n)
sim = _sim_kind(X, A, lam_c0, lam0, rng, e1, e2)
T, E = sim["Tobs"], sim["E"]
tau_i = sim["S1"] - sim["S0"]             # true CATE
print("n:", n, "| events:", int(E.sum()), "| region share:", round(float(union.mean()), 3))


## Stage 2 — Splits (50 / 25 / 25)
Discovery / validation / assessment are disjoint; assessment is never used for
selection or commitment.


In [ ]:
ii = np.arange(n)
disc, hold = train_test_split(ii, test_size=0.5, random_state=SEED, stratify=A)
val, asm = train_test_split(hold, test_size=0.5, random_state=SEED + 1, stratify=A[hold])

XD, AD, TD, ED = X[disc], A[disc], T[disc], E[disc]
XV, AV, TV, EV = X[val], A[val], T[val], E[val]
XA, AA, TA, EA = X[asm], A[asm], T[asm], E[asm]
uniA, truth = union[asm], tau_i[asm]
print("discovery/validation/assessment:", len(disc), len(val), len(asm))


## Stage 3 — Nuisances, cross-fitting, censoring, pseudo-outcome
`crossfit_nuisances` fits per-arm Cox outcome models and a conditional Cox
censoring model on 5 folds; `_capped_G` caps the IPCW weights; `pseudo_outcome`
builds the AIPW doubly-robust target (propensity e = 0.5 because randomized).


In [ ]:
muD, GD, Gmarg, fid, fcaps, _ = crossfit_nuisances(
    XD, AD, TD, ED, T_STAR, seed=SEED, return_folds=True)
G = _capped_G(muD, GD, Gmarg, fid, fcaps, "conditional", TD)
psi = pseudo_outcome(AD, TD, T_STAR, muD, G)
print("pseudo-outcome: mean", round(float(np.nanmean(psi)), 3),
      "| sd", round(float(np.nanstd(psi)), 3),
      "| max weight", round(float((1/G).max()), 2))


## Stage 4 — Rule discovery
A depth-3 CART is fit on the pseudo-outcomes (`gate11._cart`); each leaf path is
a rule, and `gate22._stats` gives each leaf's discovery mean, SE, and support.


In [ ]:
leaves = _cart(psi, XD, 3)[1]
means, ses_d, sup = _stats(psi, leaves)
print("leaves:", len(leaves))
for k, r in enumerate(leaves):
    print(f"  leaf {k}: support={int(sup[k]):4d}  mean_psi={means[k]:+.3f}")


## Stage 5 — **C300 selection**
Among leaves with **support ≥ 300**, pick the highest discovery mean
(this is the only difference from C200, which uses 200).


In [ ]:
ok = sup >= FLOOR
if ok.any():
    j = int(np.nanargmax(np.where(ok, means, -np.inf)))
    print("C300: support floor met")
else:
    j = int(np.nanargmax(means))
    print("C300: floor NOT met -> fallback to global max")

rule = " & ".join(f"x{feat} {op} {thr:+.3f}" for feat, op, thr in leaves[j]["cond"])
print("selected leaf:", j, "| support:", int(sup[j]))
print("rule (standardized space):", rule)


## Stage 6 — Validation effect + commitment
Estimate the leaf effect on the disjoint validation set; **commit only if the
one-sided 95% lower bound is above zero**, else abstain (INSUFFICIENT).


In [ ]:
p1v, p0v = phi_on_holdout(XD, AD, TD, ED, XV, AV, TV, T_STAR, seed=SEED)
dphi_v = p1v - p0v
tbl_v = _leaf_table(leaves, XV, dphi_v)
effs = np.array([t["eff"] for t in tbl_v], float)
ses = np.array([t["se"] for t in tbl_v], float)
const = float(aipw(XD, AD, TD, T_STAR, muD, G)[0])
effs = np.where(np.isfinite(effs), effs, const)

mv = eval_rule(leaves[j]["cond"], XV)
kv = int(mv.sum())
hi_eff = float(dphi_v[mv].mean()) if kv else np.nan
hi_se = float(np.sqrt(((dphi_v[mv] - hi_eff) ** 2).sum()) / kv) if kv else np.nan
committed = bool(np.isfinite(hi_eff) and np.isfinite(hi_se) and hi_eff > 1.645 * hi_se)
print(f"validation effect = {hi_eff:+.3f}  (SE {hi_se:.3f}, n={kv})")
print("COMMIT:" , "YES -> TREAT" if committed else "NO -> INSUFFICIENT")


## Stage 7 — Prediction
Assign each assessment patient the most-specific matching leaf effect; the rest
get the default ATE (`gate19._assign_effs`).


In [ ]:
use = effs.copy()
if not committed:
    use[j] = const
masks_a = [eval_rule(r["cond"], XA) for r in leaves]
tbl_a = [dict(cond=leaves[t]["cond"], mask=masks_a[t], eff=use[t], se=ses[t])
         for t in range(len(leaves))]
pred = _assign_effs(tbl_a, XA, use, const)
treated = masks_a[j] if committed else np.zeros(len(XA), bool)
print("prediction: mean", round(float(pred.mean()), 3), "| treated:", int(treated.sum()))


## Stage 8 — Evaluation (`gate28._metrics`)
Purity, coverage, capture, per-treated benefit, MAE, Spearman.


In [ ]:
def report(name, mc):
    return dict(method=name,
                purity=round(float(mc["purity"]), 3) if mc["purity"] == mc["purity"] else None,
                coverage=round(float(mc["coverage"]), 3),
                capture=round(float(mc["capture"]), 3),
                per_treated=round(float(mc["per_treated"]), 3) if mc["per_treated"] == mc["per_treated"] else None,
                mae=round(float(mc["mae"]), 4),
                spearman=round(float(mc["spearman"]), 3) if mc["spearman"] == mc["spearman"] else None)

mc_c300 = _metrics(pred, treated, truth, uniA)
print(report("C300", mc_c300))


## Stage 9 — SCRE baseline (same split)
`scre.SurvivalCausalRuleEnsemble` (RF/RSF rules + shared-basis Coxnet); alpha
chosen by validation C-index (`gate4.scre_select_alpha`). SCRE has no abstention.


In [ ]:
s = SurvivalCausalRuleEnsemble(alpha=0.5, pred_n_estimators=100,
                                progn_n_estimators=100, rule_min_support=20,
                                max_rules=200, max_rule_conditions=3,
                                max_selected_rules=10, random_state=42)
with contextlib.redirect_stdout(io.StringIO()):
    s.fit(XD, psi, np.isfinite(psi), feature_names=[f"x{i}" for i in range(XD.shape[1])],
          time=TD, event=ED.astype(bool), treatment=AD)
    pick = scre_select_alpha(s, XV, AV, TV, EV.astype(bool), T_STAR)
if pick is not None:
    s.alpha_ = s.alpha_final_ = pick[1]
    s.coef_ = pick[2]
s.t_star_ = T_STAR
with contextlib.redirect_stdout(io.StringIO()):
    tau_s = np.asarray(s.predict(XA), float)
mc_scre = _metrics(tau_s, tau_s > 0, truth, uniA)
print(report("SCRE", mc_scre))


## Stage 10 — Side-by-side comparison


In [ ]:
comparison = pd.DataFrame([report("C300", mc_c300), report("SCRE", mc_scre)])
print(comparison.to_string(index=False))
print("\nNote: C300 may abstain (commitment NO); SCRE always treats tau_hat > 0.")


## Stage 11 — Audit + real-data ACTG175 demo
`python -m ciscarl2.audit` verifies the core; `demo_c200.main()` runs the
real-data illustration (C200 on ACTG175 — C300 would need ≥300 discovery
patients, which ACTG175 does not have).


In [ ]:
# 1) core audit
import subprocess
r = subprocess.run([sys.executable, "-m", "ciscarl2.audit"], capture_output=True, text=True)
print(r.stdout[-600:] if r.stdout else r.stderr[-600:])

# 2) real-data ACTG175 illustration (C200)
from ciscarl2 import demo_c200
demo_c200.main()


## Notes / honesty
- **C300 is our extension**, not in the paper: the published model is **C200**.
  The only change is `FLOOR = 300` in Stage 5.
- Purity is only defined when the model commits (otherwise it abstains).
- ACTG175 has no observed individual effect; accuracy is measured only in simulation.
- Intervals are empirical (~93%), not nominally valid 95%.
- SCRE here uses the bundled Python reimplementation (`scre.py`), not the authors' R package.
